# Baby Step 3 — Tax Planning Exo-Brain
## Provenance, atomic claims, contradiction control, and traceability

This notebook preserves Steps 0–2 and creates a Step 3 copy. It is a synthetic educational architecture test—not tax or legal advice.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Control architecture

Every relied-upon tax-code version becomes an immutable source snapshot. Recommendation-specific atomic claims inherit source confidence. Limiting claims create explicit unresolved tensions and dependencies. The validation requires complete source → claim → recommendation → decision lineage while preserving every Step 2 restriction.


In [ ]:
import csv
import hashlib
import json
import shutil
from collections import Counter, defaultdict
from datetime import date
from pathlib import Path


STEP3_QUARTER = "2026-Q3"
STEP3_PROVENANCE_VERSION = "2026-Q3-P001"
STEP3_DATE = date(2026, 7, 21).isoformat()


def read_csv(path: Path) -> list[dict]:
    with path.open(encoding="utf-8", newline="") as stream:
        return list(csv.DictReader(stream))


def write_csv(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        raise ValueError(f"No rows supplied for {path}")
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content.rstrip() + "\n", encoding="utf-8")


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def bool_text(value) -> str:
    return str(value).strip().lower()


def source_confidence(change_sensitivity: str) -> tuple[str, float]:
    sensitivity = change_sensitivity.strip().lower()
    if sensitivity == "high":
        return "MEDIUM", 0.70
    if sensitivity == "medium":
        return "HIGH", 0.82
    return "HIGH", 0.90


def atomic_statement(code: dict) -> str:
    return (
        f"Synthetic module {code['tax_code_id']} ({code['title']}) in {code['jurisdiction']} "
        f"records numeric parameter {code['numeric_parameter']} and threshold "
        f"{code['threshold_million_local']} million local units for version {code['version']}."
    )


def apply_step3(source_vault: Path, output_vault: Path) -> dict:
    """Copy Step 2 and add immutable sources, atomic claims, and contradiction control."""
    source_vault = Path(source_vault)
    output_vault = Path(output_vault)
    required = [
        source_vault / "13_Audit" / "STEP_2_SUCCESS.md",
        source_vault / "16_Data" / "tax_codes.csv",
        source_vault / "16_Data" / "recommendations.csv",
        source_vault / "16_Data" / "recommendation_rule_links.csv",
        source_vault / "16_Data" / "recommendation_resilience.csv",
        source_vault / "16_Data" / "decision_gates.csv",
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("Step 2 source is incomplete:\n" + "\n".join(missing))

    if output_vault.exists():
        shutil.rmtree(output_vault)
    shutil.copytree(source_vault, output_vault)

    data = output_vault / "16_Data"
    tax_codes = read_csv(data / "tax_codes.csv")
    recommendations = read_csv(data / "recommendations.csv")
    rule_links = read_csv(data / "recommendation_rule_links.csv")
    resilience = read_csv(data / "recommendation_resilience.csv")
    decisions = read_csv(data / "decision_gates.csv")
    groups = read_csv(data / "conglomerates.csv")
    entities = read_csv(data / "entities.csv")
    transactions = read_csv(data / "intercompany_transactions.csv")

    if len(tax_codes) != 100 or len(recommendations) != 10 or len(rule_links) != 70:
        raise AssertionError("Step 3 requires the validated Step 2 state: 100 codes, 10 recommendations, 70 rule links.")

    code_by_id = {row["tax_code_id"]: row for row in tax_codes}
    rec_by_id = {row["recommendation_id"]: row for row in recommendations}
    resilience_by_rec = {row["recommendation_id"]: row for row in resilience}
    source_ids: dict[str, str] = {}
    source_rows: list[dict] = []

    # One immutable source snapshot for every unique tax-code version actually relied upon.
    for index, code_id in enumerate(sorted({row["tax_code_id"] for row in rule_links}), 1):
        code = code_by_id[code_id]
        candidates = sorted((output_vault / "02_Tax_Codes").glob(f"{code_id}_*.md"))
        if len(candidates) != 1:
            raise AssertionError(f"Expected one source document for {code_id}, found {len(candidates)}")
        source_path = candidates[0]
        source_id = f"SRC-{index:03d}"
        source_ids[code_id] = source_id
        confidence_label, confidence_score = source_confidence(code["change_sensitivity"])
        row = {
            "source_id": source_id,
            "tax_code_id": code_id,
            "jurisdiction_id": code["jurisdiction_id"],
            "jurisdiction": code["jurisdiction"],
            "module": code["module"],
            "source_version": code["version"],
            "effective_from": code["effective_from"],
            "source_status": code["status"],
            "change_sensitivity": code["change_sensitivity"],
            "confidence_label": confidence_label,
            "confidence_score": confidence_score,
            "source_path": source_path.relative_to(output_vault).as_posix(),
            "content_sha256": sha256(source_path),
            "captured_at": STEP3_DATE,
            "provenance_version": STEP3_PROVENANCE_VERSION,
            "synthetic": True,
        }
        source_rows.append(row)
        write_text(output_vault / "06_Sources" / f"{source_id}_{code_id}.md", f"""
---
object_type: source_record
source_id: {source_id}
tax_code_id: {code_id}
source_version: {code['version']}
content_sha256: {row['content_sha256']}
confidence: {confidence_label}
provenance_version: {STEP3_PROVENANCE_VERSION}
synthetic: true
---

# {source_id} — {code_id} source snapshot

- Canonical source: [[{source_path.stem}]]
- Jurisdiction: {code['jurisdiction']} ([[{code['jurisdiction_id']}]]).
- Module: {code['module']}.
- Version: {code['version']}.
- Effective from: {code['effective_from']}.
- Change sensitivity: {code['change_sensitivity']}.
- Confidence: **{confidence_label}** ({confidence_score:.2f}).
- SHA-256: `{row['content_sha256']}`.

This record freezes the exact synthetic source used by Step 3. A future code change must create a new version and may not overwrite this provenance record.
""")

    source_by_id = {row["source_id"]: row for row in source_rows}
    claim_rows: list[dict] = []
    claims_by_rec: defaultdict[str, list[dict]] = defaultdict(list)

    for index, link in enumerate(sorted(rule_links, key=lambda x: (x["recommendation_id"], x["relation"], int(x["sequence"]))), 1):
        code = code_by_id[link["tax_code_id"]]
        source_id = source_ids[link["tax_code_id"]]
        source = source_by_id[source_id]
        claim_id = f"CLM-{index:03d}"
        polarity = "SUPPORTS" if link["relation"] == "SUPPORTING" else "LIMITS"
        permission_effect = "MAY_SUPPORT_BOUNDED_REVIEW" if polarity == "SUPPORTS" else "NARROWS_RELIANCE_UNTIL_RESOLVED"
        row = {
            "claim_id": claim_id,
            "recommendation_id": link["recommendation_id"],
            "conglomerate_id": link["conglomerate_id"],
            "source_id": source_id,
            "tax_code_id": link["tax_code_id"],
            "source_version": link["version"],
            "module": link["module"],
            "polarity": polarity,
            "atomic_statement": atomic_statement(code),
            "confidence_label": source["confidence_label"],
            "confidence_score": source["confidence_score"],
            "permission_effect": permission_effect,
            "provenance_version": STEP3_PROVENANCE_VERSION,
            "synthetic": True,
        }
        claim_rows.append(row)
        claims_by_rec[link["recommendation_id"]].append(row)
        write_text(output_vault / "07_Atomic_Claims" / f"{claim_id}.md", f"""
---
object_type: atomic_tax_claim
claim_id: {claim_id}
recommendation_id: {link['recommendation_id']}
conglomerate_id: {link['conglomerate_id']}
source_id: {source_id}
tax_code_id: {link['tax_code_id']}
source_version: {link['version']}
polarity: {polarity}
confidence: {source['confidence_label']}
permission_effect: {permission_effect}
synthetic: true
---

# {claim_id} — {polarity.title()} {link['module']} claim

## Atomic statement

{row['atomic_statement']}

## Lineage

[[{source_id}_{link['tax_code_id']}]] → **{claim_id}** → [[{link['recommendation_id']}]] → [[{rec_by_id[link['recommendation_id']]['decision_id']}]]

## Governance effect

- Confidence: **{source['confidence_label']}** ({source['confidence_score']}).
- Permission effect: **{permission_effect}**.
- This claim is indivisible within the Step 3 schema; interpretation beyond this statement requires a new claim.
""")

    # Each of four limiting claims is explicitly paired with a supporting claim.
    contradiction_rows: list[dict] = []
    dependency_rows: list[dict] = []
    contradictions_by_rec: defaultdict[str, list[dict]] = defaultdict(list)
    contradiction_index = 1
    for rec_id in sorted(claims_by_rec):
        supporting = sorted((r for r in claims_by_rec[rec_id] if r["polarity"] == "SUPPORTS"), key=lambda x: x["claim_id"])
        limiting = sorted((r for r in claims_by_rec[rec_id] if r["polarity"] == "LIMITS"), key=lambda x: x["claim_id"])
        if len(supporting) != 3 or len(limiting) != 4:
            raise AssertionError(f"Expected 3 supporting and 4 limiting claims for {rec_id}")
        for offset, limit_claim in enumerate(limiting):
            support_claim = supporting[offset % len(supporting)]
            contradiction_id = f"CTR-{contradiction_index:03d}"
            contradiction_index += 1
            row = {
                "contradiction_id": contradiction_id,
                "recommendation_id": rec_id,
                "conglomerate_id": rec_by_id[rec_id]["conglomerate_id"],
                "supporting_claim_id": support_claim["claim_id"],
                "limiting_claim_id": limit_claim["claim_id"],
                "tension_type": "DESIGN_BENEFIT_VS_ANTI_AVOIDANCE_CONSTRAINT",
                "status": "UNRESOLVED",
                "severity": "HIGH" if limit_claim["module"] in {"CFC", "GMT"} else "MEDIUM",
                "control_effect": "NARROW_PERMISSION",
                "resolution_requirement": "Human-reviewed evidence, calculations, and a documented reconciliation",
                "provenance_version": STEP3_PROVENANCE_VERSION,
                "synthetic": True,
            }
            contradiction_rows.append(row)
            contradictions_by_rec[rec_id].append(row)
            dependency_rows.append({
                "dependency_id": f"DEP-{len(dependency_rows)+1:03d}",
                "recommendation_id": rec_id,
                "dependent_claim_id": support_claim["claim_id"],
                "constraint_claim_id": limit_claim["claim_id"],
                "contradiction_id": contradiction_id,
                "dependency_rule": "Supporting reliance is bounded by the unresolved limiting claim",
                "status": "ACTIVE",
                "synthetic": True,
            })
            write_text(output_vault / "08_Contradictions" / f"{contradiction_id}.md", f"""
---
object_type: contradiction
contradiction_id: {contradiction_id}
recommendation_id: {rec_id}
supporting_claim_id: {support_claim['claim_id']}
limiting_claim_id: {limit_claim['claim_id']}
status: UNRESOLVED
severity: {row['severity']}
control_effect: NARROW_PERMISSION
synthetic: true
---

# {contradiction_id} — unresolved design tension

- Supporting claim: [[{support_claim['claim_id']}]].
- Limiting claim: [[{limit_claim['claim_id']}]].
- Type: design benefit versus anti-avoidance constraint.
- Severity: **{row['severity']}**.
- Status: **UNRESOLVED**.

## Control

The limiting claim bounds reliance on the supporting claim. Resolution requires human-reviewed evidence, calculations, and a documented reconciliation. Silence, missing evidence, or higher uncertainty may only narrow permission.
""")

    trace_rows: list[dict] = []
    summary_rows: list[dict] = []
    for rec in recommendations:
        rec_id = rec["recommendation_id"]
        rec_claims = sorted(claims_by_rec[rec_id], key=lambda x: x["claim_id"])
        rec_contradictions = contradictions_by_rec[rec_id]
        for claim in rec_claims:
            trace_rows.append({
                "recommendation_id": rec_id,
                "conglomerate_id": rec["conglomerate_id"],
                "decision_id": rec["decision_id"],
                "claim_id": claim["claim_id"],
                "polarity": claim["polarity"],
                "source_id": claim["source_id"],
                "tax_code_id": claim["tax_code_id"],
                "source_version": claim["source_version"],
                "source_sha256": source_by_id[claim["source_id"]]["content_sha256"],
                "confidence_score": claim["confidence_score"],
                "trace_complete": True,
                "synthetic": True,
            })

        res = resilience_by_rec[rec_id]
        if res["resilience_class"] == "ROBUST":
            reliance = "BOUNDED_TRACEABLE"
            disposition = "QUALIFY_FOR_STEP_4_WITH_UNRESOLVED_TENSIONS"
        elif res["resilience_class"] == "MANAGED_FRAGILITY":
            reliance = "TRACEABLE_WITH_MITIGATIONS"
            disposition = "QUALIFY_FOR_STEP_4_EXCEPTION_REPORTING"
        else:
            reliance = "TRACEABLE_BUT_REOPENED"
            disposition = "EXCLUDE_FROM_STEP_4_DECISION_SET_PENDING_REDESIGN"
        minimum_confidence = min(float(c["confidence_score"]) for c in rec_claims)
        summary = {
            "recommendation_id": rec_id,
            "conglomerate_id": rec["conglomerate_id"],
            "conglomerate": rec["conglomerate"],
            "source_count": len({c["source_id"] for c in rec_claims}),
            "claim_count": len(rec_claims),
            "supporting_claims": sum(c["polarity"] == "SUPPORTS" for c in rec_claims),
            "limiting_claims": sum(c["polarity"] == "LIMITS" for c in rec_claims),
            "unresolved_contradictions": len(rec_contradictions),
            "minimum_claim_confidence": round(minimum_confidence, 2),
            "traceability_completeness_pct": 100.0,
            "step2_resilience_class": res["resilience_class"],
            "reliance_class": reliance,
            "step3_disposition": disposition,
            "implementation_authorized": False,
            "provenance_version": STEP3_PROVENANCE_VERSION,
            "synthetic": True,
        }
        summary_rows.append(summary)
        rec["status"] = "PROVENANCE_CONTROLLED_PENDING_HUMAN_REVIEW"
        rec["step3_provenance_version"] = STEP3_PROVENANCE_VERSION
        rec["step3_reliance_class"] = reliance
        rec["step3_disposition"] = disposition

        claim_lines = "\n".join(
            f"- [[{c['claim_id']}]] — {c['polarity']} {c['module']}; confidence {c['confidence_label']}."
            for c in rec_claims
        )
        contradiction_lines = "\n".join(
            f"- [[{c['contradiction_id']}]] — {c['severity']}; {c['status']}; permission narrows."
            for c in rec_contradictions
        )
        rec_path = output_vault / "09_Recommendations" / f"{rec_id}.md"
        rec_text = rec_path.read_text(encoding="utf-8")
        rec_text += f"""

## Step 3 provenance and claim control

- Provenance version: **{STEP3_PROVENANCE_VERSION}**
- Traceability completeness: **100%**
- Reliance class: **{reliance}**
- Step 3 disposition: **{disposition}**

### Atomic claims

{claim_lines}

### Unresolved tensions

{contradiction_lines}

Complete traceability does not resolve a substantive weakness and does not widen implementation authority.
"""
        write_text(rec_path, rec_text)

    summary_by_rec = {row["recommendation_id"]: row for row in summary_rows}
    for decision in decisions:
        summary = summary_by_rec[decision["recommendation_id"]]
        decision["status"] = "STEP_3_PROVENANCE_CONTROLLED_PENDING_HUMAN_REVIEW"
        decision["permitted_action"] = summary["step3_disposition"]
        decision["implementation_authorized"] = False
        decision["provenance_version"] = STEP3_PROVENANCE_VERSION
        decision_path = output_vault / "10_Decisions" / f"{decision['decision_id']}.md"
        decision_text = decision_path.read_text(encoding="utf-8")
        decision_text += f"""

## Step 3 evidence control

- Claims traced: {summary['claim_count']}.
- Unresolved contradictions: {summary['unresolved_contradictions']}.
- Traceability completeness: {summary['traceability_completeness_pct']}%.
- Reliance class: **{summary['reliance_class']}**.
- Permitted analytical action: **{summary['step3_disposition']}**.
- Implementation remains unauthorized.
"""
        write_text(decision_path, decision_text)

    for group in groups:
        group["recommendation_status"] = "PROVENANCE_CONTROLLED_PENDING_HUMAN_REVIEW"

    write_csv(data / "source_provenance.csv", source_rows)
    write_csv(data / "atomic_tax_claims.csv", claim_rows)
    write_csv(data / "claim_dependencies.csv", dependency_rows)
    write_csv(data / "contradictions.csv", contradiction_rows)
    write_csv(data / "source_claim_recommendation_trace.csv", trace_rows)
    write_csv(data / "recommendation_traceability_summary.csv", summary_rows)
    write_csv(data / "recommendations.csv", recommendations)
    write_csv(data / "decision_gates.csv", decisions)
    write_csv(data / "conglomerates.csv", groups)

    source_table = "\n".join(
        f"| [[{row['source_id']}_{row['tax_code_id']}]] | {row['tax_code_id']} | {row['jurisdiction']} | {row['module']} | {row['source_version']} | {row['confidence_label']} | `{row['content_sha256'][:12]}…` |"
        for row in source_rows
    )
    write_text(output_vault / "12_Reports" / "STEP_3_PROVENANCE_REGISTER.md", f"""
---
object_type: provenance_register
report_id: RPT-STEP-3-PROV
quarter: {STEP3_QUARTER}
provenance_version: {STEP3_PROVENANCE_VERSION}
status: INTERNAL_SYNTHETIC_DRAFT
synthetic: true
---

# Step 3 Provenance Register

The register freezes the 44 unique tax-code versions actually relied upon by the ten Recommendation V1 records.

| Source | Tax code | Jurisdiction | Module | Version | Confidence | Hash |
|---|---|---|---|---|---|---|
{source_table}

The remaining 56 synthetic tax-code modules stay in the governed inventory but were not relied upon by Recommendation V1. No source is silently promoted into the trace.
""")

    claim_counts = Counter(row["polarity"] for row in claim_rows)
    confidence_counts = Counter(row["confidence_label"] for row in claim_rows)
    claim_table = "\n".join(
        f"| [[{row['claim_id']}]] | [[{row['recommendation_id']}]] | [[{row['source_id']}_{row['tax_code_id']}]] | {row['module']} | {row['polarity']} | {row['confidence_label']} |"
        for row in claim_rows
    )
    write_text(output_vault / "12_Reports" / "STEP_3_ATOMIC_CLAIMS_REGISTER.md", f"""
---
object_type: atomic_claims_register
report_id: RPT-STEP-3-CLM
quarter: {STEP3_QUARTER}
provenance_version: {STEP3_PROVENANCE_VERSION}
synthetic: true
---

# Step 3 Atomic Tax Claims Register

| Claim | Recommendation | Source | Module | Polarity | Confidence |
|---|---|---|---|---|---|
{claim_table}

## Portfolio totals

- Atomic claims: {len(claim_rows)}.
- Supporting: {claim_counts['SUPPORTS']}.
- Limiting: {claim_counts['LIMITS']}.
- High-confidence: {confidence_counts['HIGH']}.
- Medium-confidence: {confidence_counts['MEDIUM']}.
""")

    contradiction_table = "\n".join(
        f"| [[{row['contradiction_id']}]] | [[{row['recommendation_id']}]] | [[{row['supporting_claim_id']}]] | [[{row['limiting_claim_id']}]] | {row['severity']} | {row['status']} |"
        for row in contradiction_rows
    )
    write_text(output_vault / "12_Reports" / "STEP_3_CONTRADICTION_REGISTER.md", f"""
---
object_type: contradiction_register
report_id: RPT-STEP-3-CTR
quarter: {STEP3_QUARTER}
provenance_version: {STEP3_PROVENANCE_VERSION}
synthetic: true
---

# Step 3 Contradiction and Tension Register

| Tension | Recommendation | Supporting claim | Limiting claim | Severity | Status |
|---|---|---|---|---|---|
{contradiction_table}

All {len(contradiction_rows)} tensions remain unresolved. This is intentional: Step 3 makes conflict visible and narrows reliance; it does not manufacture resolution.
""")

    summary_table = "\n".join(
        f"| [[{row['recommendation_id']}]] | {row['conglomerate']} | {row['source_count']} | {row['claim_count']} | {row['unresolved_contradictions']} | {row['minimum_claim_confidence']} | {row['traceability_completeness_pct']}% | {row['reliance_class']} | {row['step3_disposition']} |"
        for row in summary_rows
    )
    reliance_counts = Counter(row["reliance_class"] for row in summary_rows)
    write_text(output_vault / "12_Reports" / "STEP_3_TRACEABILITY_REGISTER.md", f"""
---
object_type: traceability_register
report_id: RPT-STEP-3-TRACE
quarter: {STEP3_QUARTER}
provenance_version: {STEP3_PROVENANCE_VERSION}
synthetic: true
---

# Step 3 Source-to-Recommendation Traceability Register

| Recommendation | Conglomerate | Sources | Claims | Tensions | Min confidence | Complete | Reliance | Disposition |
|---|---|---:|---:|---:|---:|---:|---|---|
{summary_table}

## Portfolio result

- {len(source_rows)} immutable source snapshots.
- {len(claim_rows)} atomic tax claims.
- {len(dependency_rows)} active claim dependencies.
- {len(contradiction_rows)} unresolved contradictions/tensions.
- {len(trace_rows)} complete source-claim-recommendation-decision trace rows.
- {reliance_counts['BOUNDED_TRACEABLE']} bounded traceable recommendations.
- {reliance_counts['TRACEABLE_WITH_MITIGATIONS']} traceable recommendations with mitigations.
- {reliance_counts['TRACEABLE_BUT_REOPENED']} traceable but reopened recommendation.

Cobalt Life Sciences remains excluded from the Step 4 decision set pending redesign. Traceability records a weakness; it does not cure it.
""")

    write_text(output_vault / "11_Quarterly_Updates" / STEP3_QUARTER / "STEP_3_PROVENANCE_CYCLE.md", f"""
# {STEP3_QUARTER} — Step 3 Provenance Cycle

- Source snapshots: {len(source_rows)}.
- Atomic claims: {len(claim_rows)}.
- Claim dependencies: {len(dependency_rows)}.
- Contradictions/tensions: {len(contradiction_rows)}.
- Trace rows: {len(trace_rows)}.
- Tax-code alterations: 0.
- New companies: 0.
- Recommendation V2 records: 0.
- Implementation authority: none.
""")

    write_text(output_vault / "14_Hot_Cache" / "CURRENT_STATE.md", f"""
# Current State — {STEP3_QUARTER} Provenance-Controlled Recommendation V1

- Active step: 3 of 10
- Tax-code modules: 100 (unchanged)
- Conglomerates: 10 (unchanged)
- Immutable relied-upon sources: {len(source_rows)}
- Atomic tax claims: {len(claim_rows)}
- Unresolved tensions: {len(contradiction_rows)}
- Complete trace rows: {len(trace_rows)}
- Step 4 qualified, bounded: {reliance_counts['BOUNDED_TRACEABLE']}
- Step 4 exception reporting: {reliance_counts['TRACEABLE_WITH_MITIGATIONS']}
- Reopened and excluded from decision set: {reliance_counts['TRACEABLE_BUT_REOPENED']}
- Explicit prohibition: no real tax advice, filing, transaction, communication, or restructuring
""")

    state = {
        "project": "Tax Planning Exo-Brain",
        "active_step": 3,
        "quarter": STEP3_QUARTER,
        "recommendation_version": "2026-Q3-R001",
        "stress_test_version": "2026-Q3-S001",
        "provenance_version": STEP3_PROVENANCE_VERSION,
        "counts": {
            "jurisdictions": 20,
            "tax_codes": len(tax_codes),
            "conglomerates": len(groups),
            "entities": len(entities),
            "transactions": len(transactions),
            "recommendations": len(recommendations),
            "relied_upon_source_snapshots": len(source_rows),
            "atomic_claims": len(claim_rows),
            "claim_dependencies": len(dependency_rows),
            "contradictions": len(contradiction_rows),
            "trace_rows": len(trace_rows),
            "decision_gates": len(decisions),
        },
        "permission": "INTERNAL_SCENARIO_ONLY",
        "implementation_authorized": False,
        "synthetic": True,
    }
    write_text(output_vault / "00_System" / "CURRENT_STATE.json", json.dumps(state, indent=2))

    excluded = {
        "13_Audit/STEP_3_VALIDATION.json",
        "13_Audit/STEP_3_MANIFEST.csv",
        "13_Audit/STEP_3_SUCCESS.md",
    }
    manifest_rows = []
    for path in sorted(p for p in output_vault.rglob("*") if p.is_file()):
        rel = path.relative_to(output_vault).as_posix()
        if rel in excluded:
            continue
        manifest_rows.append({"path": rel, "bytes": path.stat().st_size, "sha256": sha256(path)})
    write_csv(output_vault / "13_Audit" / "STEP_3_MANIFEST.csv", manifest_rows)

    source_hashes_match = all(
        sha256(output_vault / row["source_path"]) == row["content_sha256"] for row in source_rows
    )
    checks = {
        "step2_success_marker_present": (output_vault / "13_Audit" / "STEP_2_SUCCESS.md").exists(),
        "tax_codes_equal_100": len(tax_codes) == 100,
        "conglomerates_equal_10": len(groups) == 10,
        "recommendations_equal_10": len(recommendations) == 10,
        "rule_links_equal_70": len(rule_links) == 70,
        "unique_relied_upon_sources_equal_44": len(source_rows) == 44,
        "atomic_claims_equal_70": len(claim_rows) == 70,
        "three_supporting_claims_per_recommendation": all(
            sum(c["polarity"] == "SUPPORTS" for c in claims_by_rec[r]) == 3 for r in claims_by_rec
        ),
        "four_limiting_claims_per_recommendation": all(
            sum(c["polarity"] == "LIMITS" for c in claims_by_rec[r]) == 4 for r in claims_by_rec
        ),
        "dependencies_equal_40": len(dependency_rows) == 40,
        "contradictions_equal_40": len(contradiction_rows) == 40,
        "all_contradictions_unresolved": all(r["status"] == "UNRESOLVED" for r in contradiction_rows),
        "trace_rows_equal_70": len(trace_rows) == 70,
        "all_trace_rows_complete": all(bool_text(r["trace_complete"]) == "true" for r in trace_rows),
        "source_hashes_match": source_hashes_match,
        "all_claim_source_versions_match": all(
            c["source_version"] == source_by_id[c["source_id"]]["source_version"] for c in claim_rows
        ),
        "all_recommendations_have_100pct_traceability": all(
            float(r["traceability_completeness_pct"]) == 100.0 for r in summary_rows
        ),
        "cobalt_remains_reopened": summary_by_rec["REC-003-V001"]["reliance_class"] == "TRACEABLE_BUT_REOPENED",
        "decision_gates_equal_10": len(decisions) == 10,
        "implementation_never_authorized": not any(bool_text(r["implementation_authorized"]) == "true" for r in decisions),
        "no_tax_code_versions_changed": all(r["version"] == "2026-Q3-V001" for r in tax_codes),
        "no_new_companies": len(groups) == 10,
    }
    validation = {
        "step": 3,
        "date": STEP3_DATE,
        "quarter": STEP3_QUARTER,
        "provenance_version": STEP3_PROVENANCE_VERSION,
        "checks": checks,
        "counts": state["counts"],
        "claim_polarity": dict(claim_counts),
        "claim_confidence": dict(confidence_counts),
        "reliance_classes": dict(reliance_counts),
        "status": "PASS" if all(checks.values()) else "FAIL",
    }
    write_text(output_vault / "13_Audit" / "STEP_3_VALIDATION.json", json.dumps(validation, indent=2))
    if validation["status"] != "PASS":
        failed = [key for key, ok in checks.items() if not ok]
        raise AssertionError("Step 3 validation failed: " + ", ".join(failed))
    write_text(output_vault / "13_Audit" / "STEP_3_SUCCESS.md", f"""
# Step 3 Validation: PASS

Validated {len(source_rows)} immutable source snapshots, {len(claim_rows)} atomic tax claims, {len(dependency_rows)} active dependencies, {len(contradiction_rows)} unresolved tensions, and {len(trace_rows)} complete lineage rows. Cobalt Life Sciences remains reopened; no implementation is authorized.
""")
    return validation


In [ ]:
from pathlib import Path
PROJECT = Path('/content/drive/MyDrive/Tax_Planning_ExoBrain_Project')
SOURCE_VAULT = PROJECT / 'Step_2' / 'Tax_Planning_ExoBrain_Vault'
OUTPUT_VAULT = PROJECT / 'Step_3' / 'Tax_Planning_ExoBrain_Vault'
validation = apply_step3(SOURCE_VAULT, OUTPUT_VAULT)
print(json.dumps(validation, indent=2))
print(f'\nStep 3 vault created at: {OUTPUT_VAULT}')


## Expected result

Validation must be `PASS`: 44 unique relied-upon source snapshots, 70 atomic claims, 40 dependencies, 40 unresolved tensions, 70 complete trace rows, 100 unchanged tax-code modules, and 10 unchanged conglomerates. Cobalt Life Sciences remains reopened. No implementation is authorized.
